# SEN12MS GAN Reproducibility Training

GAN menggunakan discriminator conditional 29 channel yang menilai raw prediction; blended output digunakan untuk evaluasi.

In [ ]:
import os
import re
import math
import random
import zipfile
import json
from datetime import datetime
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from scipy.ndimage import gaussian_filter
except Exception:
    gaussian_filter = None

try:
    import tifffile
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tifffile"])
    import tifffile

from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split, WeightedRandomSampler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("Torch:", torch.__version__)

## 1. Konfigurasi GAN

In [ ]:
@dataclass
class CFG:
    DATASET_NAME: str = "asiaWest_n"
    EXPERIMENT_NAME: str = "gan_reproducibility_training"
    LOCAL_ROOT: Path = Path(r"C:/Users/Vulpolish/Downloads/Tugas Akhir/asiaWest_n")
    KAGGLE_INPUT: Path = Path("/kaggle/input")
    WORK_DATA: Path = Path("/kaggle/working/data")
    OUT_DIR: Path = Path("/kaggle/working/gan_reproducibility_training") if Path("/kaggle/working").exists() else Path("gan_reproducibility_training")
    BATCH_SIZE: int = 4
    NUM_WORKERS: int = 2
    EPOCHS: int = 5
    GAN_EPOCHS: int = 20
    LR: float = 5e-6
    G_LR: float = 1e-5
    D_LR: float = 5e-6
    TRAIN_RATIO: float = 0.80
    VAL_RATIO: float = 0.10
    RUN_TRAINING: bool = False
    RUN_GAN_FINETUNE: bool = True
    REQUIRE_T4_FOR_GAN: bool = True
    USE_S1: bool = True
    BASE_CHANNELS: int = 32
    CLOUD_THRESHOLD: float = 0.22
    MASK_LOW: float = 0.08
    MASK_HIGH: float = 0.20
    TARGET_SHARPNESS: float = 10.0
    MAX_TRAIN_SAMPLES: int = 0
    HEAVY_CLOUD_OVERSAMPLE: bool = True
    HEAVY_CLOUD_THRESHOLD: float = 0.30
    HEAVY_PIXEL_THRESHOLD: float = 0.66
    HEAVY_LOSS_BOOST: float = 2.5
    HEAVY_BINARY_BOOST: float = 2.0
    LAMBDA_SSIM: float = 0.01
    LAMBDA_ADV: float = 0.001
    LAMBDA_FEATURE_MATCHING: float = 0.05
    REFINE_DILATION_PIXELS: int = 2
    REFINE_GAUSSIAN_SIGMA: float = 0.8
    MIN_IMPROVEMENT_EPS: float = 1e-7
    REFERENCE_RGB_MASK_MAE: float = 0.014853816471680152
    REFERENCE_BAND13_MASK_MAE: float = 0.017478999864943844
    REFERENCE_RGB_GLOBAL_MAE: float = 0.006883921497313711
    REFERENCE_BAND13_GLOBAL_MAE: float = 0.008587756912660447
    BASELINE_RGB_MASK_MAE: float = 0.015089
    BASELINE_BAND13_MASK_MAE: float = 0.018551
    EXPECTED_TEST_SAMPLE_SHA256: str = "6151bee4138eb76aee5555f55f4ff6c77d5ed7019725f9fd3f1e712849318eb3"
    BASELINE_CHECKPOINT_FILENAME: str = "best_multitemporal_resunet_hardmask.pth"
    REFERENCE_CHECKPOINT_FILENAME: str = "best_metric_safe_resunet_finetune.pth"

CFG.OUT_DIR.mkdir(parents=True, exist_ok=True)
CFG.WORK_DATA.mkdir(parents=True, exist_ok=True)


def assert_training_accelerator_ready():
    if not CFG.RUN_GAN_FINETUNE:
        return
    if not torch.cuda.is_available():
        raise RuntimeError("GAN membutuhkan GPU. Pilih Accelerator GPU T4, restart session, lalu Run All ulang.")
    gpu_name = torch.cuda.get_device_name(0)
    print("CUDA device:", gpu_name)
    print("CUDA capability:", torch.cuda.get_device_capability(0))
    if CFG.REQUIRE_T4_FOR_GAN and "T4" not in gpu_name.upper():
        raise RuntimeError(f"GAN diset untuk GPU T4, tetapi device saat ini: {gpu_name}. Pilih Accelerator GPU T4 lalu restart session.")


assert_training_accelerator_ready()
CFG


## 2. Cari Dataset dan Unzip Jika Perlu

In [ ]:
def unzip_if_needed(input_root: Path, work_root: Path, dataset_name: str = "asiaWest_n") -> None:
    if not input_root.exists():
        return
    if (work_root / dataset_name).exists():
        return
    zips = sorted(input_root.rglob(f"{dataset_name}.zip")) + sorted(input_root.rglob("*.zip"))
    if not zips:
        return
    zip_path = zips[0]
    print("Unzipping:", zip_path)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(work_root)


def find_dataset_root() -> Path:
    candidates = []
    if CFG.LOCAL_ROOT.exists():
        candidates.append(CFG.LOCAL_ROOT)
    if CFG.KAGGLE_INPUT.exists():
        candidates += list(CFG.KAGGLE_INPUT.rglob(CFG.DATASET_NAME))
    candidates += list(CFG.WORK_DATA.rglob(CFG.DATASET_NAME))
    for c in candidates:
        if c.is_dir() and any(c.glob("ROIs*")):
            return c
    unzip_if_needed(CFG.KAGGLE_INPUT, CFG.WORK_DATA, CFG.DATASET_NAME)
    candidates = list(CFG.WORK_DATA.rglob(CFG.DATASET_NAME))
    for c in candidates:
        if c.is_dir() and any(c.glob("ROIs*")):
            return c
    print("Kaggle input folders:")
    if CFG.KAGGLE_INPUT.exists():
        for p in CFG.KAGGLE_INPUT.glob("*"):
            print(" -", p)
    raise FileNotFoundError("Dataset asiaWest_n belum ditemukan. Upload/attach asiaWest_n.zip atau folder asiaWest_n dulu.")

DATA_ROOT = find_dataset_root()
print("DATA_ROOT:", DATA_ROOT)
print("Top folders:", [p.name for p in DATA_ROOT.iterdir() if p.is_dir()][:10])

## 3. Index SEN12MS-CR-TS

In [ ]:
SEN12_RE = re.compile(
    r"^(?P<sensor>s[12])_(?P<roi_group>ROIs\d+)_(?P<roi_id>\d+)_ImgNo_"
    r"(?P<time_id>\d+)_(?P<date>\d{4}-\d{2}-\d{2})_patch_(?P<patch_id>\d+)\.tif$",
    re.IGNORECASE,
)

@dataclass(frozen=True)
class SEN12Sample:
    roi_group: str
    roi_id: str
    patch_id: int
    s1: Dict[int, Path]
    s2: Dict[int, Path]
    dates: Dict[Tuple[str, int], str]

    @property
    def sample_id(self) -> str:
        return f"{self.roi_group}_{self.roi_id}_patch_{self.patch_id}"


def build_index(root: Path) -> List[SEN12Sample]:
    grouped = {}
    for path in sorted(root.rglob("*.tif")):
        m = SEN12_RE.match(path.name)
        if not m:
            continue
        gd = m.groupdict()
        key = (gd["roi_group"], gd["roi_id"], int(gd["patch_id"]))
        sensor = gd["sensor"].upper()
        time_id = int(gd["time_id"])
        item = grouped.setdefault(key, {"S1": {}, "S2": {}, "dates": {}})
        item[sensor][time_id] = path
        item["dates"][(sensor, time_id)] = gd["date"]

    samples = []
    for (roi_group, roi_id, patch_id), item in sorted(grouped.items()):
        if set(item["S1"]) == {0,1,2,3} and set(item["S2"]) == {0,1,2,3}:
            samples.append(SEN12Sample(roi_group, roi_id, patch_id, dict(sorted(item["S1"].items())), dict(sorted(item["S2"].items())), dict(sorted(item["dates"].items()))))
    return samples

samples = build_index(DATA_ROOT)
print("Complete multi-temporal samples:", len(samples))
print(samples[0])

In [ ]:
rows = []
for s in samples:
    rows.append({
        "sample_id": s.sample_id,
        "roi_group": s.roi_group,
        "roi_id": s.roi_id,
        "patch_id": s.patch_id,
        "s1_dates": ", ".join(s.dates[("S1", t)] for t in range(4)),
        "s2_dates": ", ".join(s.dates[("S2", t)] for t in range(4)),
    })
index_df = pd.DataFrame(rows)
display(index_df.head())
display(index_df.groupby(["roi_group", "roi_id"]).size().reset_index(name="n_patches"))
index_df.to_csv(CFG.OUT_DIR / "sen12ms_asiawest_index.csv", index=False)

## 4. Reader dan Target Pseudo Ground Truth

In [ ]:
def read_tif(path: Path) -> np.ndarray:
    arr = tifffile.imread(path)
    if arr.ndim == 2:
        arr = arr[np.newaxis, :, :]
    if arr.ndim == 3 and arr.shape[-1] in {2, 3, 4, 13}:
        arr = np.moveaxis(arr, -1, 0)
    return arr.astype(np.float32, copy=False)


def robust_scale(arr: np.ndarray, p_low=2, p_high=98) -> np.ndarray:
    lo, hi = np.nanpercentile(arr, [p_low, p_high])
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        return np.zeros_like(arr, dtype=np.float32)
    return np.clip((arr - lo) / (hi - lo), 0, 1).astype(np.float32)


def normalize_s2(arr: np.ndarray) -> np.ndarray:
    return np.clip(arr / 10000.0, 0, 1).astype(np.float32)


def normalize_s1(arr: np.ndarray) -> np.ndarray:
    # S1 biasanya dB sekitar -35..5. Skala konservatif ke [0,1].
    return np.clip((arr + 35.0) / 40.0, 0, 1).astype(np.float32)


def s2_rgb(s2: np.ndarray) -> np.ndarray:
    return robust_scale(np.stack([s2[3], s2[2], s2[1]], axis=-1))


def s1_vv_vh_rgb(s1: np.ndarray) -> np.ndarray:
    vv = robust_scale(s1[0])
    vh = robust_scale(s1[1])
    return np.stack([vv, vh, np.abs(vv-vh)], axis=-1)


def spectral_cloud_probability_s2(s2_norm: np.ndarray) -> np.ndarray:
    # SEN12MS band order umum: B1,B2,B3,B4,B5,B6,B7,B8,B8A,B9,B10,B11,B12.
    blue = s2_norm[1]
    green = s2_norm[2]
    red = s2_norm[3]
    nir = s2_norm[7] if s2_norm.shape[0] > 7 else red
    cirrus = s2_norm[10] if s2_norm.shape[0] > 10 else blue
    swir1 = s2_norm[11] if s2_norm.shape[0] > 11 else nir
    swir2 = s2_norm[12] if s2_norm.shape[0] > 12 else swir1
    rgb = np.stack([red, green, blue], axis=0)
    brightness = rgb.mean(axis=0)
    whiteness = 1.0 - rgb.std(axis=0)
    ndvi = (nir - red) / (nir + red + 1e-6)
    ndsi = (green - swir1) / (green + swir1 + 1e-6)
    visible_cloud = brightness * np.clip(whiteness, 0, 1)
    high_band = 0.60 * cirrus + 0.25 * swir1 + 0.15 * swir2
    vegetation_penalty = np.clip((ndvi - 0.20) / 0.35, 0, 1)
    snow_penalty = np.clip((ndsi - 0.35) / 0.35, 0, 1) * np.clip((brightness - 0.20) / 0.45, 0, 1)
    prob = 0.62 * visible_cloud + 0.38 * high_band
    prob = prob * (1.0 - 0.35 * vegetation_penalty) * (1.0 - 0.25 * snow_penalty)
    return np.clip(prob, 0, 1).astype(np.float32)


def temporal_cloud_probabilities_s2(stack_norm: np.ndarray) -> np.ndarray:
    # stack_norm: T,C,H,W. Temporal contrast mengurangi false positive pada tanah/salju cerah stabil.
    spectral = np.stack([spectral_cloud_probability_s2(stack_norm[t]) for t in range(stack_norm.shape[0])], axis=0)
    blue = stack_norm[:, 1]
    green = stack_norm[:, 2]
    red = stack_norm[:, 3]
    brightness = (red + green + blue) / 3.0
    temporal_floor = np.percentile(brightness, 20, axis=0)
    temporal_median = np.median(brightness, axis=0)
    brighter_than_clear = np.clip((brightness - temporal_floor) / 0.18, 0, 1)
    brighter_than_median = np.clip((brightness - temporal_median) / 0.12, 0, 1)
    temporal = 0.65 * brighter_than_clear + 0.35 * brighter_than_median
    probs = 0.58 * spectral + 0.42 * temporal
    return np.clip(probs, 0, 1).astype(np.float32)


def refine_soft_mask(mask: np.ndarray) -> np.ndarray:
    mask = np.clip(mask, 0, 1).astype(np.float32)
    if gaussian_filter is not None:
        mask = gaussian_filter(mask, sigma=1.0)
    return np.clip(mask, 0, 1).astype(np.float32)


def soft_cloud_mask_from_prob(prob: np.ndarray) -> np.ndarray:
    mask = np.clip((prob - CFG.MASK_LOW) / (CFG.MASK_HIGH - CFG.MASK_LOW + 1e-6), 0, 1)
    return refine_soft_mask(mask)


def cloud_probability_s2(s2: np.ndarray) -> np.ndarray:
    arr = normalize_s2(s2) if s2.max() > 1.5 else s2.astype(np.float32)
    return spectral_cloud_probability_s2(arr)


def cloud_score_s2(s2: np.ndarray) -> float:
    return float(cloud_probability_s2(s2).mean())

sample = samples[0]
s2 = read_tif(sample.s2[0])
s1 = read_tif(sample.s1[0])
s2_stack = np.stack([normalize_s2(read_tif(sample.s2[t])) for t in range(4)], axis=0)
probs = temporal_cloud_probabilities_s2(s2_stack)
print("S2 shape/dtype/min/max:", s2.shape, s2.dtype, float(s2.min()), float(s2.max()))
print("S1 shape/dtype/min/max:", s1.shape, s1.dtype, float(s1.min()), float(s1.max()))
print("Temporal cloud prob t0 mean/max:", float(probs[0].mean()), float(probs[0].max()))
print("Soft mask t0 mean/max:", float(soft_cloud_mask_from_prob(probs[0]).mean()), float(soft_cloud_mask_from_prob(probs[0]).max()))

In [ ]:
def temporal_median(sample: SEN12Sample) -> np.ndarray:
    stack = np.stack([normalize_s2(read_tif(sample.s2[t])) for t in range(4)], axis=0)
    return np.median(stack, axis=0).astype(np.float32)


def clear_pixel_composite(sample: SEN12Sample) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    stack = np.stack([normalize_s2(read_tif(sample.s2[t])) for t in range(4)], axis=0)  # T,C,H,W
    probs = temporal_cloud_probabilities_s2(stack)                                    # T,H,W
    best_map = np.argmin(probs, axis=0).astype(np.int64)
    weights = np.exp(-CFG.TARGET_SHARPNESS * probs).astype(np.float32) + 1e-5
    weights = weights / weights.sum(axis=0, keepdims=True)
    target_weighted = (stack * weights[:, None]).sum(axis=0).astype(np.float32)
    target_best = np.zeros_like(target_weighted)
    for t in range(stack.shape[0]):
        pick = best_map == t
        target_best[:, pick] = stack[t, :, pick].T
    # Campur best-pixel dan weighted composite: cukup tajam, tapi lebih halus dari hard best-map.
    target = (0.45 * target_best + 0.55 * target_weighted).astype(np.float32)
    cloud_t0 = soft_cloud_mask_from_prob(probs[0])
    return target, cloud_t0, probs, best_map


def clear_pixel_composite_from_stack(stack: np.ndarray, probs: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    best_map = np.argmin(probs, axis=0).astype(np.int64)
    weights = np.exp(-CFG.TARGET_SHARPNESS * probs).astype(np.float32) + 1e-5
    weights = weights / weights.sum(axis=0, keepdims=True)
    target_weighted = (stack * weights[:, None]).sum(axis=0).astype(np.float32)
    target_best = np.zeros_like(target_weighted)
    for t in range(stack.shape[0]):
        pick = best_map == t
        target_best[:, pick] = stack[t, :, pick].T
    target = (0.45 * target_best + 0.55 * target_weighted).astype(np.float32)
    cloud_t0 = soft_cloud_mask_from_prob(probs[0])
    return target, cloud_t0, best_map


def weighted_clear_composite(sample: SEN12Sample) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    target, cloud_t0, probs, _ = clear_pixel_composite(sample)
    return target, cloud_t0, probs


def best_time_s2(sample: SEN12Sample) -> Tuple[int, np.ndarray, List[float]]:
    arrs = [normalize_s2(read_tif(sample.s2[t])) for t in range(4)]
    probs = temporal_cloud_probabilities_s2(np.stack(arrs, axis=0))
    scores = [float(probs[t].mean()) for t in range(4)]
    best_t = int(np.argmin(scores))
    return best_t, arrs[best_t], scores


def np_mae(a, b, mask=None):
    err = np.abs(a - b)
    if mask is not None and mask.sum() > 0:
        err = err * mask[None]
        return float(err.sum() / (mask.sum() * a.shape[0] + 1e-8))
    return float(err.mean())


def plot_baseline(sample: SEN12Sample):
    best_t, best_arr, scores = best_time_s2(sample)
    target, cloud_t0, probs, best_map = clear_pixel_composite(sample)
    t0 = normalize_s2(read_tif(sample.s2[0]))
    median_arr = temporal_median(sample)
    fig, axes = plt.subplots(2, 6, figsize=(21, 7))
    for t in range(4):
        axes[0, t].imshow(s2_rgb(read_tif(sample.s2[t])))
        axes[0, t].set_title(f"S2 t{t} | score={scores[t]:.3f}")
        axes[0, t].axis("off")
    axes[0, 4].imshow(cloud_t0, cmap="gray", vmin=0, vmax=1)
    axes[0, 4].set_title("auto cloud mask t0")
    axes[0, 4].axis("off")
    axes[0, 5].imshow(best_map, cmap="viridis", vmin=0, vmax=3)
    axes[0, 5].set_title("best date map")
    axes[0, 5].axis("off")
    panels = [(t0, "cloudy t0"), (best_arr, f"best whole t{best_t}"), (median_arr, "median"), (target, "pseudo ground truth")]
    for ax, (arr, title) in zip(axes[1, :4], panels):
        ax.imshow(s2_rgb(arr))
        ax.set_title(title)
        ax.axis("off")
    axes[1, 4].imshow(probs[0], cmap="magma", vmin=0, vmax=1)
    axes[1, 4].set_title("cloud prob t0")
    axes[1, 4].axis("off")
    axes[1, 5].axis("off")
    plt.suptitle(sample.sample_id)
    plt.tight_layout()
    plt.show()
    print({
        "t0_mae_to_pseudo_gt": np_mae(t0, target),
        "best_mae_to_pseudo_gt": np_mae(best_arr, target),
        "median_mae_to_pseudo_gt": np_mae(median_arr, target),
        "t0_cloud_mae": np_mae(t0, target, cloud_t0),
        "best_cloud_mae": np_mae(best_arr, target, cloud_t0),
        "mask_mean": float(cloud_t0.mean()),
    })

for s in random.sample(samples, min(5, len(samples))):
    plot_baseline(s)

## 5. Dataset PyTorch Multi-Temporal dan Split Sama

In [ ]:
class SEN12TemporalDataset(Dataset):
    def __init__(self, samples: List[SEN12Sample], target_mode="weighted"):
        self.samples = samples
        self.target_mode = target_mode

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        s2_list = [normalize_s2(read_tif(s.s2[t])) for t in range(4)]
        temporal_probs = temporal_cloud_probabilities_s2(np.stack(s2_list, axis=0))
        cloud_probs = [temporal_probs[t][None] for t in range(4)]
        inputs = [*s2_list, *cloud_probs]
        if CFG.USE_S1:
            s1_list = [normalize_s1(read_tif(s.s1[t])) for t in range(4)]
            inputs.extend(s1_list)
        x = np.concatenate(inputs, axis=0).astype(np.float32)

        if self.target_mode == "weighted":
            target, cloud_mask, _ = clear_pixel_composite_from_stack(np.stack(s2_list, axis=0), temporal_probs)
        elif self.target_mode == "median":
            target = temporal_median(s)
            cloud_mask = soft_cloud_mask_from_prob(temporal_cloud_probabilities_s2(np.stack(s2_list, axis=0))[0])
        elif self.target_mode == "best_time":
            _, target, _ = best_time_s2(s)
            cloud_mask = soft_cloud_mask_from_prob(temporal_cloud_probabilities_s2(np.stack(s2_list, axis=0))[0])
        else:
            raise ValueError(self.target_mode)
        return {
            "x": torch.from_numpy(x),
            "y": torch.from_numpy(target.astype(np.float32)),
            "cloud_mask": torch.from_numpy(cloud_mask[None].astype(np.float32)),
            "t0": torch.from_numpy(s2_list[0].astype(np.float32)),
            "sample_id": s.sample_id,
        }

work_samples = samples
if CFG.MAX_TRAIN_SAMPLES and len(samples) > CFG.MAX_TRAIN_SAMPLES:
    work_samples = sorted(random.Random(SEED).sample(samples, CFG.MAX_TRAIN_SAMPLES), key=lambda s: s.sample_id)

dataset = SEN12TemporalDataset(work_samples, target_mode="weighted")
item = dataset[0]
print(item["x"].shape, item["y"].shape, item["cloud_mask"].shape, item["sample_id"])
INPUT_CHANNELS = item["x"].shape[0]
print("INPUT_CHANNELS:", INPUT_CHANNELS)

In [ ]:
import hashlib

SPLIT_SEED = 42
n = len(dataset)
n_train = int(n * CFG.TRAIN_RATIO)
n_val = int(n * CFG.VAL_RATIO)
n_test = n - n_train - n_val
split_generator = torch.Generator().manual_seed(SPLIT_SEED)
train_ds, val_ds, test_ds = random_split(dataset, [n_train, n_val, n_test], generator=split_generator)


def subset_base_indices(subset):
    return list(subset.indices) if hasattr(subset, "indices") else list(range(len(subset)))


def split_manifest(subset, split_name):
    base_indices = subset_base_indices(subset)
    base_dataset = subset.dataset if hasattr(subset, "dataset") else subset
    rows = []
    for local_i, base_i in enumerate(base_indices):
        if hasattr(base_dataset, "samples"):
            sample_id = base_dataset.samples[int(base_i)].sample_id
        else:
            sample_id = base_dataset[int(base_i)]["sample_id"]
        rows.append({
            "split": split_name,
            "local_index": int(local_i),
            "base_index": int(base_i),
            "sample_id": sample_id,
            "split_seed": int(SPLIT_SEED),
        })
    return pd.DataFrame(rows)


split_df = pd.concat([
    split_manifest(train_ds, "train"),
    split_manifest(val_ds, "val"),
    split_manifest(test_ds, "test"),
], ignore_index=True)

split_sets = {name: set(group["base_index"]) for name, group in split_df.groupby("split")}
assert split_sets["train"].isdisjoint(split_sets["val"])
assert split_sets["train"].isdisjoint(split_sets["test"])
assert split_sets["val"].isdisjoint(split_sets["test"])
assert sum(len(v) for v in split_sets.values()) == len(dataset)

test_sample_ids = split_df.loc[split_df["split"] == "test", "sample_id"].tolist()
test_sample_sha256 = hashlib.sha256("\n".join(test_sample_ids).encode("utf-8")).hexdigest()
split_df.to_csv(CFG.OUT_DIR / "sen12ms_split_manifest_seed42.csv", index=False)
split_df.loc[split_df["split"] == "test"].to_csv(CFG.OUT_DIR / "final_test_sample_ids.csv", index=False)
print("split:", len(train_ds), len(val_ds), len(test_ds))
print("split seed:", SPLIT_SEED)
print("test sample sha256:", test_sample_sha256)
display(split_df.groupby("split").size().reset_index(name="n_samples"))


def compute_subset_cloud_means(subset, name="train"):
    """Hitung rata-rata auto mask per sample untuk oversampling dan bucket analysis."""
    base_indices = subset_base_indices(subset)
    base_dataset = subset.dataset if hasattr(subset, "dataset") else subset
    rows = []
    for local_i, base_i in enumerate(tqdm(base_indices, desc=f"cloud mean scan {name}", leave=False)):
        item = base_dataset[int(base_i)]
        rows.append({
            "local_index": local_i,
            "base_index": int(base_i),
            "sample_id": item["sample_id"],
            "cloud_mean": float(item["cloud_mask"].mean()),
        })
    return pd.DataFrame(rows)


train_cloud_df = compute_subset_cloud_means(train_ds, "train")
val_cloud_df = compute_subset_cloud_means(val_ds, "val")
test_cloud_df = compute_subset_cloud_means(test_ds, "test")
train_cloud_df.to_csv(CFG.OUT_DIR / "train_cloud_sampling_weights.csv", index=False)
val_cloud_df.to_csv(CFG.OUT_DIR / "val_cloud_distribution.csv", index=False)
test_cloud_df.to_csv(CFG.OUT_DIR / "test_cloud_distribution.csv", index=False)


def make_train_sampler(cloud_df):
    cloud = cloud_df["cloud_mean"].to_numpy(dtype=np.float32)
    # Sample berawan sedang/berat muncul lebih sering agar model tidak hanya bagus di awan ringan.
    weights = 1.0 + 2.0 * cloud
    weights += 3.0 * (cloud >= CFG.HEAVY_CLOUD_THRESHOLD).astype(np.float32)
    weights += 2.0 * (cloud >= 0.45).astype(np.float32)
    return WeightedRandomSampler(
        weights=torch.as_tensor(weights, dtype=torch.double),
        num_samples=len(weights),
        replacement=True,
        generator=torch.Generator().manual_seed(SPLIT_SEED),
    ), weights


loader_kwargs = dict(num_workers=CFG.NUM_WORKERS, pin_memory=torch.cuda.is_available())
if CFG.NUM_WORKERS > 0:
    loader_kwargs.update(prefetch_factor=2, persistent_workers=True)

if CFG.HEAVY_CLOUD_OVERSAMPLE:
    train_sampler, train_sampling_weights = make_train_sampler(train_cloud_df)
    train_loader = DataLoader(train_ds, batch_size=CFG.BATCH_SIZE, sampler=train_sampler, **loader_kwargs)
else:
    train_sampling_weights = np.ones(len(train_ds), dtype=np.float32)
    train_loader = DataLoader(train_ds, batch_size=CFG.BATCH_SIZE, shuffle=True, **loader_kwargs)

val_loader = DataLoader(val_ds, batch_size=CFG.BATCH_SIZE, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_ds, batch_size=CFG.BATCH_SIZE, shuffle=False, **loader_kwargs)

print("train cloud mean summary:")
display(train_cloud_df["cloud_mean"].describe().to_frame().T)
print("test cloud mean summary:")
display(test_cloud_df["cloud_mean"].describe().to_frame().T)
print("heavy train samples:", int((train_cloud_df["cloud_mean"] >= CFG.HEAVY_CLOUD_THRESHOLD).sum()))
print("oversampling:", CFG.HEAVY_CLOUD_OVERSAMPLE, "weight min/max:", float(np.min(train_sampling_weights)), float(np.max(train_sampling_weights)))
batch = next(iter(train_loader))
print("batch x/y:", batch["x"].shape, batch["y"].shape)

## 6. Arsitektur Generator ResUNet Sama Dengan V5

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
        )
        self.skip = nn.Conv2d(in_ch, out_ch, 1, bias=False) if in_ch != out_ch else nn.Identity()
        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.act(self.net(x) + self.skip(x))

class Down(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(nn.MaxPool2d(2), ConvBlock(in_ch, out_ch))
    def forward(self, x):
        return self.net(x)

class Up(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up = nn.Sequential(
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
        self.conv = ConvBlock(out_ch + skip_ch, out_ch)
    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        return self.conv(torch.cat([x, skip], dim=1))

class MultiTemporalResUNet(nn.Module):
    def __init__(self, in_channels=64, out_channels=13, base=32):
        super().__init__()
        self.inc = ConvBlock(in_channels, base)
        self.down1 = Down(base, base*2)
        self.down2 = Down(base*2, base*4)
        self.down3 = Down(base*4, base*8)
        self.up1 = Up(base*8, base*4, base*4)
        self.up2 = Up(base*4, base*2, base*2)
        self.up3 = Up(base*2, base, base)
        self.outc = nn.Conv2d(base, out_channels, 1)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x = self.up1(x4, x3)
        x = self.up2(x, x2)
        x = self.up3(x, x1)
        return torch.sigmoid(self.outc(x))

model = MultiTemporalResUNet(in_channels=INPUT_CHANNELS, base=CFG.BASE_CHANNELS).to(DEVICE)
with torch.no_grad():
    yhat = model(batch["x"].to(DEVICE))
print("output:", yhat.shape)
print("params:", sum(p.numel() for p in model.parameters()))

## 7. Helper Loss

In [ ]:
def mae(pred, target):
    return torch.mean(torch.abs(pred - target))

def masked_mae(pred, target, mask):
    err = torch.abs(pred - target) * mask
    return err.sum() / (mask.sum() * pred.shape[1] + 1e-8)

def weighted_masked_mae(pred, target, mask, weight):
    effective = mask * weight
    err = torch.abs(pred - target) * effective
    return err.sum() / (effective.sum() * pred.shape[1] + 1e-8)

def bucket_mae(pred, target, bucket_mask):
    denom = bucket_mask.sum() * pred.shape[1]
    if float(denom.detach().cpu()) < 1.0:
        return torch.tensor(float("nan"), device=pred.device)
    return (torch.abs(pred - target) * bucket_mask).sum() / (denom + 1e-8)

def gradient_mae(pred, target, mask):
    dx_p = pred[:, :, :, 1:] - pred[:, :, :, :-1]
    dx_t = target[:, :, :, 1:] - target[:, :, :, :-1]
    mx = mask[:, :, :, 1:]
    dy_p = pred[:, :, 1:, :] - pred[:, :, :-1, :]
    dy_t = target[:, :, 1:, :] - target[:, :, :-1, :]
    my = mask[:, :, 1:, :]
    return 0.5 * (masked_mae(dx_p, dx_t, mx) + masked_mae(dy_p, dy_t, my))

def rmse(pred, target):
    return torch.sqrt(torch.mean((pred - target) ** 2) + 1e-8)

def psnr(pred, target, max_val=1.0):
    mse = torch.mean((pred - target) ** 2)
    return 20 * torch.log10(torch.tensor(max_val, device=pred.device)) - 10 * torch.log10(mse + 1e-8)

def apply_copy_outside_mask(raw_pred, t0, mask):
    # Soft gate: pixel dengan mask 0 copy persis dari cloudy t0.
    # Pixel makin berawan makin boleh diubah model.
    gate = torch.clamp(mask, 0.0, 1.0)
    return gate * raw_pred + (1.0 - gate) * t0

def heavy_cloud_weight(mask):
    return 1.0 + CFG.HEAVY_LOSS_BOOST * mask + CFG.HEAVY_BINARY_BOOST * (mask >= CFG.HEAVY_PIXEL_THRESHOLD).float()

def cloud_bucket_masks(mask):
    light = ((mask > 0.05) & (mask < 0.33)).float()
    medium = ((mask >= 0.33) & (mask < CFG.HEAVY_PIXEL_THRESHOLD)).float()
    heavy = (mask >= CFG.HEAVY_PIXEL_THRESHOLD).float()
    return light, medium, heavy

def scalar_or_none(x):
    value = float(x.detach().cpu())
    return None if math.isnan(value) else value

optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG.EPOCHS, eta_min=CFG.LR * 0.1)
BEST_PATH = CFG.OUT_DIR / "best_multitemporal_resunet_hardmask.pth"
LAST_PATH = CFG.OUT_DIR / "last_multitemporal_resunet_hardmask.pth"
HISTORY_PATH = CFG.OUT_DIR / "training_history.csv"
SUMMARY_PATH = CFG.OUT_DIR / "metrics_summary.json"
BEST_METRIC_CANDIDATES = ["val_rgb_mask_mae", "val_band13_mask_mae", "val_model_cloud_mae", "val_loss"]

def cfg_to_checkpoint_dict():
    out = {}
    for name in CFG.__dataclass_fields__:
        value = getattr(CFG, name)
        out[name] = str(value) if isinstance(value, Path) else value
    return out

def select_best_validation_metric(row):
    for name in BEST_METRIC_CANDIDATES:
        value = row.get(name)
        if value is not None and np.isfinite(value):
            return name, float(value)
    raise ValueError("No finite validation metric found for checkpoint selection.")

def build_baseline_checkpoint(epoch, row, best_metric_name, best_validation_metric):
    return {
        "epoch": int(epoch),
        "model": model.state_dict(),
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "metrics": row,
        "validation_metrics": row,
        "best_metric_name": best_metric_name,
        "best_validation_metric": float(best_validation_metric),
        "config": cfg_to_checkpoint_dict(),
        "split_seed": int(SPLIT_SEED),
        "test_sample_sha256": test_sample_sha256,
    }

@torch.no_grad()
def evaluate(loader, label="val"):
    model.eval()
    logs = []
    iterator = tqdm(loader, desc=f"evaluate {label}", leave=False)
    for batch in iterator:
        x = batch["x"].to(DEVICE)
        y = batch["y"].to(DEVICE)
        mask = batch["cloud_mask"].to(DEVICE)
        t0 = batch["t0"].to(DEVICE)
        raw = model(x)
        pred = apply_copy_outside_mask(raw, t0, mask)
        light_m, medium_m, heavy_m = cloud_bucket_masks(mask)
        logs.append({
            "model_mae": float(mae(pred, y).cpu()),
            "model_cloud_mae": float(masked_mae(pred, y, mask).cpu()),
            "model_light_cloud_mae": scalar_or_none(bucket_mae(pred, y, light_m)),
            "model_medium_cloud_mae": scalar_or_none(bucket_mae(pred, y, medium_m)),
            "model_heavy_cloud_mae": scalar_or_none(bucket_mae(pred, y, heavy_m)),
            "raw_cloud_mae": float(masked_mae(raw, y, mask).cpu()),
            "t0_mae": float(mae(t0, y).cpu()),
            "t0_cloud_mae": float(masked_mae(t0, y, mask).cpu()),
            "t0_light_cloud_mae": scalar_or_none(bucket_mae(t0, y, light_m)),
            "t0_medium_cloud_mae": scalar_or_none(bucket_mae(t0, y, medium_m)),
            "t0_heavy_cloud_mae": scalar_or_none(bucket_mae(t0, y, heavy_m)),
            "rmse": float(rmse(pred, y).cpu()),
            "psnr": float(psnr(pred, y).cpu()),
            "cloud_fraction": float(mask.mean().cpu()),
            "heavy_fraction": float(heavy_m.mean().cpu()),
        })
    return pd.DataFrame(logs).mean(numeric_only=True).to_dict()

def train_one_epoch(loader, epoch, total_epochs):
    model.train()
    losses = []
    cloud_losses = []
    heavy_losses = []
    iterator = tqdm(loader, desc=f"epoch {epoch:03d}/{total_epochs:03d}", leave=True)
    for batch in iterator:
        x = batch["x"].to(DEVICE)
        y = batch["y"].to(DEVICE)
        mask = batch["cloud_mask"].to(DEVICE)
        t0 = batch["t0"].to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        raw = model(x)
        pred = apply_copy_outside_mask(raw, t0, mask)
        weight = heavy_cloud_weight(mask)
        whole = F.l1_loss(pred, y)
        cloud_plain = masked_mae(pred, y, mask)
        cloud_weighted = weighted_masked_mae(pred, y, mask, weight)
        raw_cloud_weighted = weighted_masked_mae(raw, y, mask, weight)
        copy_penalty = masked_mae(pred, t0, 1.0 - mask)
        grad = gradient_mae(pred, y, mask)
        _, _, heavy_m = cloud_bucket_masks(mask)
        heavy_only = bucket_mae(pred, y, heavy_m)
        heavy_term = torch.where(torch.isnan(heavy_only), cloud_plain, heavy_only)
        # Fokus utama: area awan berat, tetap menjaga luar mask copy dari input.
        loss = (
            0.15 * whole
            + 1.15 * cloud_weighted
            + 0.20 * raw_cloud_weighted
            + 0.25 * copy_penalty
            + 0.10 * grad
            + 0.15 * heavy_term
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        losses.append(float(loss.detach().cpu()))
        cloud_losses.append(float(cloud_plain.detach().cpu()))
        heavy_losses.append(0.0 if bool(torch.isnan(heavy_only).detach().cpu()) else float(heavy_only.detach().cpu()))
        iterator.set_postfix(loss=np.mean(losses), cloud=np.mean(cloud_losses), heavy=np.mean(heavy_losses), lr=optimizer.param_groups[0]["lr"])
    return float(np.mean(losses)), float(np.mean(cloud_losses)), float(np.mean(heavy_losses))

def save_training_plot(history_df):
    if history_df.empty:
        return
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].plot(history_df["epoch"], history_df["train_loss"], marker="o", label="train loss")
    axes[0].plot(history_df["epoch"], history_df["val_model_mae"], marker="o", label="val model MAE")
    axes[0].plot(history_df["epoch"], history_df["val_t0_mae"], linestyle="--", label="baseline t0 MAE")
    axes[0].set_title("Whole image MAE")
    axes[0].set_xlabel("epoch")
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()
    axes[1].plot(history_df["epoch"], history_df["val_model_cloud_mae"], marker="o", label="model cloud MAE")
    axes[1].plot(history_df["epoch"], history_df["val_t0_cloud_mae"], linestyle="--", label="baseline cloud MAE")
    if "val_model_heavy_cloud_mae" in history_df:
        axes[1].plot(history_df["epoch"], history_df["val_model_heavy_cloud_mae"], marker=".", label="model heavy MAE")
    if "val_t0_heavy_cloud_mae" in history_df:
        axes[1].plot(history_df["epoch"], history_df["val_t0_heavy_cloud_mae"], linestyle=":", label="baseline heavy MAE")
    axes[1].set_title("Cloud bucket MAE")
    axes[1].set_xlabel("epoch")
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()
    plt.tight_layout()
    out = CFG.OUT_DIR / "training_curves.png"
    plt.savefig(out, dpi=160, bbox_inches="tight")
    print("Saved curve:", out)
    plt.show()

print("=" * 70)
print("TRAINING CONFIG")
print("device:", DEVICE)
print("samples:", len(dataset), "train/val/test:", len(train_ds), len(val_ds), len(test_ds))
print("epochs:", CFG.EPOCHS, "batch:", CFG.BATCH_SIZE, "workers:", CFG.NUM_WORKERS, "input_channels:", INPUT_CHANNELS)
print("strategy: hard/soft copy outside mask; heavy-cloud weighted loss; heavy-cloud oversampling")
print("heavy thresholds: sample >=", CFG.HEAVY_CLOUD_THRESHOLD, "pixel >=", CFG.HEAVY_PIXEL_THRESHOLD)
print("output:", CFG.OUT_DIR)
print("=" * 70)

if CFG.RUN_TRAINING:
    baseline_metrics = evaluate(val_loader, label="val baseline")
    print("BASELINE CHECK")
    print(json.dumps(baseline_metrics, indent=2))

history = []
best_cloud_mae = float("inf")
best_validation_metric = float("inf")
best_metric_name = None
best_epoch = 0
best_row = None
if CFG.RUN_TRAINING:
    for epoch in range(1, CFG.EPOCHS + 1):
        print(f"\n===== EPOCH {epoch}/{CFG.EPOCHS} | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} =====")
        train_loss, train_cloud_loss, train_heavy_loss = train_one_epoch(train_loader, epoch, CFG.EPOCHS)
        scheduler.step()
        val_metrics = evaluate(val_loader, label=f"val epoch {epoch}")
        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_cloud_loss": train_cloud_loss,
            "train_heavy_loss": train_heavy_loss,
            "lr": optimizer.param_groups[0]["lr"],
            **{f"val_{k}": v for k, v in val_metrics.items()},
        }
        history.append(row)
        hist_df = pd.DataFrame(history)
        hist_df.to_csv(HISTORY_PATH, index=False)
        metric_name, metric_value = select_best_validation_metric(row)
        torch.save(build_baseline_checkpoint(epoch, row, metric_name, metric_value), LAST_PATH)
        improved = metric_value < best_validation_metric
        if improved:
            best_validation_metric = metric_value
            best_metric_name = metric_name
            best_cloud_mae = val_metrics["model_cloud_mae"]
            best_epoch = epoch
            best_row = row
            torch.save(build_baseline_checkpoint(epoch, row, best_metric_name, best_validation_metric), BEST_PATH)
        gap = val_metrics["model_cloud_mae"] - val_metrics["t0_cloud_mae"]
        heavy_gap = val_metrics.get("model_heavy_cloud_mae", np.nan) - val_metrics.get("t0_heavy_cloud_mae", np.nan)
        print("EPOCH SUMMARY")
        print(json.dumps(row, indent=2))
        print(f"best_epoch={best_epoch} best_metric={best_metric_name} best_value={best_validation_metric:.6f} best_cloud_mae={best_cloud_mae:.6f} gap_vs_t0={gap:.6f} heavy_gap_vs_t0={heavy_gap:.6f} improved={improved}")
        if epoch % 5 == 0 or epoch == CFG.EPOCHS:
            save_training_plot(hist_df)
    hist_df = pd.DataFrame(history)
    display(hist_df)
    save_training_plot(hist_df)
    final_row = hist_df.iloc[-1].to_dict()
    summary = {
        "best_epoch": int(best_epoch),
        "best_metric_name": best_metric_name,
        "best_validation_metric": float(best_validation_metric),
        "best_cloud_mae": float(best_cloud_mae),
        "best_model_heavy_cloud_mae": None if best_row is None or pd.isna(best_row.get("val_model_heavy_cloud_mae")) else float(best_row.get("val_model_heavy_cloud_mae")),
        "baseline_t0_cloud_mae": float(final_row["val_t0_cloud_mae"]),
        "baseline_t0_heavy_cloud_mae": None if pd.isna(final_row.get("val_t0_heavy_cloud_mae")) else float(final_row.get("val_t0_heavy_cloud_mae")),
        "final_model_cloud_mae": float(final_row["val_model_cloud_mae"]),
        "final_model_heavy_cloud_mae": None if pd.isna(final_row.get("val_model_heavy_cloud_mae")) else float(final_row.get("val_model_heavy_cloud_mae")),
        "is_better_than_t0_baseline": bool(best_cloud_mae < float(final_row["val_t0_cloud_mae"])),
        "strategy": "heavy-cloud weighted loss + heavy-cloud oversampling + bucket metrics",
        "history_path": str(HISTORY_PATH),
        "best_path": str(BEST_PATH),
    }
    SUMMARY_PATH.write_text(json.dumps(summary, indent=2), encoding="utf-8")
    print("FINAL SUMMARY")
    print(json.dumps(summary, indent=2))
else:
    print("RUN_TRAINING=False. EDA dan forward pass saja.")

## 8. GAN Reproducibility Training Fine-Tune

In [ ]:
# GAN: conditional raw-D fine-tune dari checkpoint Baseline, bukan training dari nol.
# Generator = MultiTemporalResUNet yang load best_multitemporal_resunet_hardmask.pth dari kernel Baseline.
# Discriminator conditional menerima 29 channel:
# condition 16 ch = cloudy S2 t0 13 + mask t0 1 + SAR t0 VV/VH 2
# image 13 ch = generated output atau pseudo ground truth.

GAN_MODEL_NAME = "gan_reproducibility_training"
RGB_BAND_INDICES = (3, 2, 1)
BASELINE_CHECKPOINT_NAME = CFG.BASELINE_CHECKPOINT_FILENAME
EPOCH0_PATH = CFG.OUT_DIR / "epoch0_baseline_init.pth"
BEST_BLENDED_PATH = CFG.OUT_DIR / "best_val_blended_cloud_mae.pth"
BEST_RAW_PATH = CFG.OUT_DIR / "best_val_raw_cloud_mae.pth"
GAN_LAST_PATH = CFG.OUT_DIR / "last_gan.pth"
TRAINING_HISTORY_PATH = CFG.OUT_DIR / "training_history.csv"
TRAINING_SUMMARY_PATH = CFG.OUT_DIR / "gan_training_summary.json"
BEST_PATH = BEST_BLENDED_PATH


def checkpoint_cfg_dict():
    out = {}
    for name in CFG.__dataclass_fields__:
        value = getattr(CFG, name)
        out[name] = str(value) if isinstance(value, Path) else value
    return out


def _json_safe(value):
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating, float)):
        value = float(value)
        return value if np.isfinite(value) else None
    if isinstance(value, dict):
        return {k: _json_safe(v) for k, v in value.items()}
    if isinstance(value, list):
        return [_json_safe(v) for v in value]
    return value


def state_dict_from_checkpoint(checkpoint):
    if isinstance(checkpoint, dict):
        for key in ("model_state_dict", "model", "generator_state_dict"):
            if key in checkpoint:
                return checkpoint[key]
    return checkpoint


def metric_safe_ssim_loss(pred, target, max_val=1.0):
    pred = torch.clamp(pred, 0.0, 1.0)
    target = torch.clamp(target, 0.0, 1.0)
    dims = (2, 3)
    c1 = (0.01 * max_val) ** 2
    c2 = (0.03 * max_val) ** 2
    mu_x = pred.mean(dim=dims, keepdim=True)
    mu_y = target.mean(dim=dims, keepdim=True)
    var_x = ((pred - mu_x) ** 2).mean(dim=dims, keepdim=True)
    var_y = ((target - mu_y) ** 2).mean(dim=dims, keepdim=True)
    cov_xy = ((pred - mu_x) * (target - mu_y)).mean(dim=dims, keepdim=True)
    ssim = ((2 * mu_x * mu_y + c1) * (2 * cov_xy + c2)) / ((mu_x ** 2 + mu_y ** 2 + c1) * (var_x + var_y + c2) + 1e-8)
    return 1.0 - ssim.mean()


class ConditionalPatchDiscriminator(nn.Module):
    def __init__(self, in_channels=29, base=32):
        super().__init__()
        sn = nn.utils.spectral_norm
        self.blocks = nn.ModuleList([
            nn.Sequential(sn(nn.Conv2d(in_channels, base, 4, stride=2, padding=1)), nn.LeakyReLU(0.2, inplace=True)),
            nn.Sequential(sn(nn.Conv2d(base, base * 2, 4, stride=2, padding=1)), nn.LeakyReLU(0.2, inplace=True)),
            nn.Sequential(sn(nn.Conv2d(base * 2, base * 4, 4, stride=2, padding=1)), nn.LeakyReLU(0.2, inplace=True)),
        ])
        self.head = sn(nn.Conv2d(base * 4, 1, 3, padding=1))

    def forward(self, condition, image, return_features=False):
        d_input_channels = condition.shape[1] + image.shape[1]
        assert d_input_channels == 29, f"Expected D input 29 channel, got {d_input_channels}"
        z = torch.cat([condition, image], dim=1)
        features = []
        for block in self.blocks:
            z = block(z)
            features.append(z)
        logits = self.head(z)
        if return_features:
            return logits, features
        return logits


def make_gan_condition(x, cloud_mask):
    cloudy_s2_t0 = x[:, 0:13]
    mask_t0 = cloud_mask
    sar_t0 = x[:, 56:58]
    condition = torch.cat([cloudy_s2_t0, mask_t0, sar_t0], dim=1)
    assert condition.shape[1] == 16, f"Expected condition 16 channel, got {condition.shape[1]}"
    return condition


def set_requires_grad(module, enabled):
    for param in module.parameters():
        param.requires_grad_(enabled)


def feature_matching_loss(fake_features, real_features):
    terms = [F.l1_loss(fake, real.detach()) for fake, real in zip(fake_features, real_features)]
    return torch.stack(terms).mean()


def resolve_baseline_checkpoint_path():
    candidates = []
    if Path("/kaggle/input").exists():
        candidates.extend(sorted(Path("/kaggle/input").rglob(BASELINE_CHECKPOINT_NAME)))
    candidates.extend([
        Path("kaggle_sen12_baseline_latest_output_retry") / "sen12ms_outputs" / BASELINE_CHECKPOINT_NAME,
        Path("kaggle_sen12_baseline_latest_output") / "sen12ms_outputs" / BASELINE_CHECKPOINT_NAME,
    ])

    def rank(path):
        text = str(path).replace("\\", "/").lower()
        if "sen12ms-cr-ts-cloud-removal-eda-baseline-unet" in text:
            return 0
        if "kaggle_sen12_baseline_latest_output_retry" in text:
            return 1
        return 2

    found = [p for p in candidates if p.exists()]
    if found:
        return sorted(found, key=lambda p: (rank(p), str(p)))[0], candidates
    searched = "\n".join(f" - {p}" for p in candidates)
    raise FileNotFoundError(f"Checkpoint Baseline {BASELINE_CHECKPOINT_NAME} tidak ditemukan.\nPath yang dicek:\n{searched}")


def load_baseline_generator_state(target_model):
    path, searched = resolve_baseline_checkpoint_path()
    checkpoint = torch.load(path, map_location=DEVICE)
    target_model.load_state_dict(state_dict_from_checkpoint(checkpoint), strict=True)
    print("Loaded Baseline generator checkpoint:", path)
    if isinstance(checkpoint, dict):
        print("Baseline checkpoint epoch:", checkpoint.get("epoch"))
        print("Baseline checkpoint best metric:", checkpoint.get("best_metric_name"), checkpoint.get("best_validation_metric"))
    return checkpoint, path, searched


def reconstruction_loss_dominant(raw, y, mask, t0):
    pred = apply_copy_outside_mask(raw, t0, mask)
    weight = heavy_cloud_weight(mask)
    whole = F.l1_loss(pred, y)
    cloud_plain = masked_mae(pred, y, mask)
    cloud_weighted = weighted_masked_mae(pred, y, mask, weight)
    raw_cloud_weighted = weighted_masked_mae(raw, y, mask, weight)
    preservation_loss = masked_mae(pred, t0, 1.0 - mask)
    grad = gradient_mae(pred, y, mask)
    _, _, heavy_m = cloud_bucket_masks(mask)
    heavy_only = bucket_mae(pred, y, heavy_m)
    heavy_term = torch.where(torch.isnan(heavy_only), cloud_plain, heavy_only)
    ssim_term = metric_safe_ssim_loss(pred[:, RGB_BAND_INDICES], y[:, RGB_BAND_INDICES])
    reconstruction_loss = 0.20 * whole + 1.20 * cloud_weighted + 0.20 * raw_cloud_weighted + 0.20 * preservation_loss + 0.10 * grad + 0.10 * heavy_term + CFG.LAMBDA_SSIM * ssim_term
    return reconstruction_loss, {
        "reconstruction_loss": reconstruction_loss,
        "cloud_plain": cloud_plain,
        "cloud_weighted": cloud_weighted,
        "raw_cloud_weighted": raw_cloud_weighted,
        "ssim_loss": ssim_term,
        "heavy_term": heavy_term,
    }


@torch.no_grad()
def evaluate_gan(loader, label="val"):
    model.eval()
    logs = []
    for batch in tqdm(loader, desc=f"evaluate GAN {label}", leave=False):
        x = batch["x"].to(DEVICE)
        y = batch["y"].to(DEVICE)
        mask = batch["cloud_mask"].to(DEVICE)
        t0 = batch["t0"].to(DEVICE)
        raw_prediction = model(x)
        blended = apply_copy_outside_mask(raw_prediction, t0, mask)
        _, _, heavy_m = cloud_bucket_masks(mask)
        logs.append({
            "raw_model_mae": float(mae(raw_prediction, y).cpu()),
            "raw_model_cloud_mae": float(masked_mae(raw_prediction, y, mask).cpu()),
            "model_mae": float(mae(blended, y).cpu()),
            "model_cloud_mae": float(masked_mae(blended, y, mask).cpu()),
            "rgb_mask_mae": float(masked_mae(blended[:, RGB_BAND_INDICES], y[:, RGB_BAND_INDICES], mask).cpu()),
            "band13_mask_mae": float(masked_mae(blended, y, mask).cpu()),
            "heavy_cloud_mae": scalar_or_none(bucket_mae(blended, y, heavy_m)),
            "t0_cloud_mae": float(masked_mae(t0, y, mask).cpu()),
            "cloud_fraction": float(mask.mean().cpu()),
            "heavy_fraction": float(heavy_m.mean().cpu()),
        })
    return pd.DataFrame(logs).mean(numeric_only=True).to_dict()


def build_gan_checkpoint(epoch, row, best_val_model_cloud_mae, discriminator):
    return {
        "epoch": int(epoch),
        "model_state_dict": model.state_dict(),
        "generator_state_dict": model.state_dict(),
        "discriminator_state_dict": None if discriminator is None else discriminator.state_dict(),
        "metrics": row,
        "best_metric_name": "val_model_cloud_mae",
        "best_validation_metric": float(best_val_model_cloud_mae),
        "config": checkpoint_cfg_dict(),
        "split_seed": int(SPLIT_SEED),
        "test_sample_sha256": test_sample_sha256,
        "baseline_checkpoint": str(baseline_checkpoint_path),
        "experiment_name": CFG.EXPERIMENT_NAME,
        "training_from_scratch": False,
    }


def train_one_epoch_gan(loader, discriminator, g_optimizer, d_optimizer, epoch, total_epochs):
    model.train()
    discriminator.train()
    logs = []
    for batch in tqdm(loader, desc=f"GAN epoch {epoch:03d}/{total_epochs:03d}", leave=True):
        x = batch["x"].to(DEVICE)
        y = batch["y"].to(DEVICE)
        mask = batch["cloud_mask"].to(DEVICE)
        t0 = batch["t0"].to(DEVICE)
        condition = make_gan_condition(x, mask)

        set_requires_grad(discriminator, True)
        d_optimizer.zero_grad(set_to_none=True)
        with torch.no_grad():
            raw_prediction = model(x)
            fake_detached = raw_prediction.detach()
        # out_fake = D(concat(condition, raw_prediction.detach()))
        real_logits = discriminator(condition, y)
        fake_logits = discriminator(condition, fake_detached)
        d_loss = 0.5 * (F.softplus(-real_logits).mean() + F.softplus(fake_logits).mean())
        d_loss.backward()
        torch.nn.utils.clip_grad_norm_(discriminator.parameters(), 1.0)
        d_optimizer.step()

        set_requires_grad(discriminator, False)
        g_optimizer.zero_grad(set_to_none=True)
        raw_prediction = model(x)
        # out_fake_G = D(concat(condition, raw_prediction))
        fake_logits_g, fake_features = discriminator(condition, raw_prediction, return_features=True)
        with torch.no_grad():
            _, real_features = discriminator(condition, y, return_features=True)
        rec_loss, rec_logs = reconstruction_loss_dominant(raw_prediction, y, mask, t0)
        adv_loss = F.softplus(-fake_logits_g).mean()
        fm_loss = feature_matching_loss(fake_features, real_features)
        g_loss = rec_loss + CFG.LAMBDA_ADV * adv_loss + CFG.LAMBDA_FEATURE_MATCHING * fm_loss
        g_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
        g_optimizer.step()
        set_requires_grad(discriminator, True)

        logs.append({
            "g_loss": float(g_loss.detach().cpu()),
            "d_loss": float(d_loss.detach().cpu()),
            "reconstruction_loss": float(rec_loss.detach().cpu()),
            "adv_loss": float(adv_loss.detach().cpu()),
            "feature_matching_loss": float(fm_loss.detach().cpu()),
            "cloud_plain": float(rec_logs["cloud_plain"].detach().cpu()),
            "ssim_loss": float(rec_logs["ssim_loss"].detach().cpu()),
        })
    return pd.DataFrame(logs).mean(numeric_only=True).to_dict()


print("=" * 70)
print("GAN CONDITIONAL GAN FINE-TUNE CONFIG")
print("device:", DEVICE)
print("experiment:", CFG.EXPERIMENT_NAME)
print("epochs:", CFG.GAN_EPOCHS, "g_lr:", CFG.G_LR, "d_lr:", CFG.D_LR)
print("lambda_adv:", CFG.LAMBDA_ADV, "lambda_fm:", CFG.LAMBDA_FEATURE_MATCHING, "lambda_ssim:", CFG.LAMBDA_SSIM)
print("samples:", len(dataset), "train/val/test:", len(train_ds), len(val_ds), len(test_ds))
print("test_sample_sha256:", test_sample_sha256)
assert test_sample_sha256 == CFG.EXPECTED_TEST_SAMPLE_SHA256, f"Unexpected test hash: {test_sample_sha256}"
print("=" * 70)

baseline_checkpoint_info, baseline_checkpoint_path, baseline_checkpoint_searched_paths = load_baseline_generator_state(model)
baseline_eval_checkpoint_path = baseline_checkpoint_path
baseline_reference_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

discriminator = ConditionalPatchDiscriminator(in_channels=29).to(DEVICE)
g_optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.G_LR, weight_decay=1e-5)
d_optimizer = torch.optim.AdamW(discriminator.parameters(), lr=CFG.D_LR, weight_decay=1e-5)
g_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(g_optimizer, T_max=max(1, CFG.GAN_EPOCHS), eta_min=CFG.G_LR * 0.25)
d_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(d_optimizer, T_max=max(1, CFG.GAN_EPOCHS), eta_min=CFG.D_LR * 0.25)

initial_val_metrics = evaluate_gan(val_loader, label="initial baseline")
initial_row = {"epoch": 0, "train_g_loss": None, "train_d_loss": None, "train_adv_loss": None, "train_feature_matching_loss": None, "train_reconstruction_loss": None, "lr_g": CFG.G_LR, "lr_d": CFG.D_LR, **{f"val_{k}": v for k, v in initial_val_metrics.items()}}
best_val_blended_cloud_mae = float(initial_row["val_model_cloud_mae"])
best_val_raw_cloud_mae = float(initial_row["val_raw_model_cloud_mae"])
best_blended_epoch = 0
best_raw_epoch = 0
best_epoch = 0
best_row = initial_row
torch.save(build_gan_checkpoint(0, initial_row, best_val_blended_cloud_mae, discriminator=None), EPOCH0_PATH)
torch.save(build_gan_checkpoint(0, initial_row, best_val_blended_cloud_mae, discriminator=None), BEST_BLENDED_PATH)
torch.save(build_gan_checkpoint(0, initial_row, best_val_raw_cloud_mae, discriminator=None), BEST_RAW_PATH)
history = [initial_row]
print("INITIAL BASELINE VALIDATION")
print(json.dumps(initial_row, indent=2))

if CFG.RUN_GAN_FINETUNE:
    for epoch in range(1, CFG.GAN_EPOCHS + 1):
        print(f"\n===== GAN REPRODUCIBILITY TRAINING EPOCH {epoch}/{CFG.GAN_EPOCHS} | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} =====")
        train_logs = train_one_epoch_gan(train_loader, discriminator, g_optimizer, d_optimizer, epoch, CFG.GAN_EPOCHS)
        g_scheduler.step()
        d_scheduler.step()
        val_metrics = evaluate_gan(val_loader, label=f"val epoch {epoch}")
        row = {
            "epoch": epoch,
            "train_g_loss": train_logs.get("g_loss"),
            "train_d_loss": train_logs.get("d_loss"),
            "train_adv_loss": train_logs.get("adv_loss"),
            "train_feature_matching_loss": train_logs.get("feature_matching_loss"),
            "train_reconstruction_loss": train_logs.get("reconstruction_loss"),
            "lr_g": g_optimizer.param_groups[0]["lr"],
            "lr_d": d_optimizer.param_groups[0]["lr"],
            **{f"val_{k}": v for k, v in val_metrics.items()},
        }
        history.append(row)
        current_val_blended_cloud_mae = float(row["val_model_cloud_mae"])
        current_val_raw_cloud_mae = float(row["val_raw_model_cloud_mae"])
        print(json.dumps(row, indent=2))
        if current_val_blended_cloud_mae < best_val_blended_cloud_mae - CFG.MIN_IMPROVEMENT_EPS:
            best_val_blended_cloud_mae = current_val_blended_cloud_mae
            best_blended_epoch = epoch
            best_epoch = epoch
            best_row = row
            torch.save(build_gan_checkpoint(epoch, row, best_val_blended_cloud_mae, discriminator), BEST_BLENDED_PATH)
            print(f"  ==> Saved best blended checkpoint epoch={epoch} val_blended_cloud_mae={best_val_blended_cloud_mae:.6f}")
        if current_val_raw_cloud_mae < best_val_raw_cloud_mae - CFG.MIN_IMPROVEMENT_EPS:
            best_val_raw_cloud_mae = current_val_raw_cloud_mae
            best_raw_epoch = epoch
            torch.save(build_gan_checkpoint(epoch, row, best_val_raw_cloud_mae, discriminator), BEST_RAW_PATH)
            print(f"  ==> Saved best raw checkpoint epoch={epoch} val_raw_cloud_mae={best_val_raw_cloud_mae:.6f}")

torch.save(build_gan_checkpoint(history[-1]["epoch"], history[-1], best_val_blended_cloud_mae, discriminator), GAN_LAST_PATH)
history_df = pd.DataFrame(history)
history_df.to_csv(TRAINING_HISTORY_PATH, index=False)
summary = {
    "experiment_name": CFG.EXPERIMENT_NAME,
    "training_from_scratch": False,
    "baseline_checkpoint": str(baseline_checkpoint_path),
    "epoch0_checkpoint": str(EPOCH0_PATH),
    "best_blended_checkpoint": str(BEST_BLENDED_PATH),
    "best_raw_checkpoint": str(BEST_RAW_PATH),
    "last_checkpoint": str(GAN_LAST_PATH),
    "best_metric_name": "val_model_cloud_mae",
    "best_val_blended_cloud_mae": best_val_blended_cloud_mae,
    "best_val_raw_cloud_mae": best_val_raw_cloud_mae,
    "best_blended_epoch": int(best_blended_epoch),
    "best_raw_epoch": int(best_raw_epoch),
    "best_epoch": int(best_epoch),
    "epochs_requested": int(CFG.GAN_EPOCHS),
    "test_sample_sha256": test_sample_sha256,
    "config": checkpoint_cfg_dict(),
}
TRAINING_SUMMARY_PATH.write_text(json.dumps(_json_safe(summary), indent=2), encoding="utf-8")
print("Saved training history:", TRAINING_HISTORY_PATH)
print("Saved training summary:", TRAINING_SUMMARY_PATH)
print("Saved epoch0 checkpoint:", EPOCH0_PATH)
print("Saved best blended checkpoint:", BEST_BLENDED_PATH)
print("Saved best raw checkpoint:", BEST_RAW_PATH)


## 9. Evaluasi Final Test Loader

In [ ]:
# Final evaluation GAN pada test_loader yang sama, tanpa refined mask.

REPORT_METRIC_COLUMNS = [
    "rgb_global_mae", "rgb_global_rmse", "rgb_global_psnr", "rgb_global_ssim", "rgb_global_pearson",
    "rgb_mask_mae", "rgb_mask_rmse", "rgb_mask_psnr", "rgb_mask_ssim", "rgb_mask_pearson",
    "band13_global_mae", "band13_global_rmse", "band13_global_psnr", "band13_global_pearson",
    "band13_mask_mae", "band13_mask_rmse", "band13_mask_psnr", "band13_mask_pearson",
]
METRIC_EPS = 1e-8
EVAL_MASK_THRESHOLD = 0.5


def _as_float_np(arr):
    return np.asarray(arr, dtype=np.float64)


def _psnr_from_mse(mse, max_val=1.0):
    if mse <= METRIC_EPS:
        return float("inf")
    return float(20.0 * np.log10(max_val) - 10.0 * np.log10(mse))


def _pearson_np(a, b):
    a = _as_float_np(a).ravel()
    b = _as_float_np(b).ravel()
    if float(a.std()) <= METRIC_EPS or float(b.std()) <= METRIC_EPS:
        return float("nan")
    return float(np.corrcoef(a, b)[0, 1])


def _ssim_np(a, b, max_val=1.0):
    a = _as_float_np(a)
    b = _as_float_np(b)
    c1 = (0.01 * max_val) ** 2
    c2 = (0.03 * max_val) ** 2
    mu_a, mu_b = float(a.mean()), float(b.mean())
    var_a, var_b = float(a.var()), float(b.var())
    cov = float(((a - mu_a) * (b - mu_b)).mean())
    return float(((2 * mu_a * mu_b + c1) * (2 * cov + c2)) / ((mu_a ** 2 + mu_b ** 2 + c1) * (var_a + var_b + c2) + 1e-8))


def _binary_mask(mask, threshold=EVAL_MASK_THRESHOLD):
    mask = _as_float_np(mask)
    return (mask > threshold).astype(np.float64)


def _weighted_mae(diff, weights):
    denom = float(weights.sum())
    return float("nan") if denom <= METRIC_EPS else float((np.abs(diff) * weights).sum() / denom)


def _weighted_mse(diff, weights):
    denom = float(weights.sum())
    return float("nan") if denom <= METRIC_EPS else float(((diff ** 2) * weights).sum() / denom)


def _split_hash(sample_ids):
    return hashlib.sha256("\n".join(map(str, sample_ids)).encode("utf-8")).hexdigest()


def _json_safe(value):
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating, float)):
        value = float(value)
        return value if np.isfinite(value) else None
    if isinstance(value, dict):
        return {k: _json_safe(v) for k, v in value.items()}
    if isinstance(value, list):
        return [_json_safe(v) for v in value]
    return value


def _masked_stat_values(pred, target, weights):
    keep = weights > 0.5
    if int(keep.sum()) == 0:
        return None, None
    return pred[keep], target[keep]


def _per_sample_report_row(method, pred_s2, target_s2, mask, sample_id, split_name, checkpoint_path):
    pred_s2 = np.clip(_as_float_np(pred_s2), 0.0, 1.0)
    target_s2 = np.clip(_as_float_np(target_s2), 0.0, 1.0)
    mask_bin = _binary_mask(mask, threshold=EVAL_MASK_THRESHOLD)
    pred_rgb = pred_s2[:, RGB_BAND_INDICES]
    target_rgb = target_s2[:, RGB_BAND_INDICES]
    rgb_diff = pred_rgb - target_rgb
    band_diff = pred_s2 - target_s2
    rgb_mse = float(np.mean(rgb_diff ** 2))
    band_mse = float(np.mean(band_diff ** 2))
    rgb_weights = mask_bin.repeat(3, axis=1)
    band_weights = mask_bin.repeat(13, axis=1)
    rgb_mask_mse = _weighted_mse(rgb_diff, rgb_weights)
    band_mask_mse = _weighted_mse(band_diff, band_weights)
    rgb_pm, rgb_tm = _masked_stat_values(pred_rgb, target_rgb, rgb_weights)
    band_pm, band_tm = _masked_stat_values(pred_s2, target_s2, band_weights)
    return {
        "method": method,
        "evaluation_split": split_name,
        "split_seed": int(SPLIT_SEED),
        "sample_id": str(sample_id),
        "n_samples": 1,
        "test_sample_sha256": "",
        "checkpoint": str(checkpoint_path) if checkpoint_path else "",
        "mask_threshold": float(EVAL_MASK_THRESHOLD),
        "mask_pixel_count": int(mask_bin.sum()),
        "rgb_global_mae": float(np.mean(np.abs(rgb_diff))),
        "rgb_global_rmse": float(np.sqrt(rgb_mse)),
        "rgb_global_psnr": _psnr_from_mse(rgb_mse),
        "rgb_global_ssim": _ssim_np(pred_rgb[0], target_rgb[0]),
        "rgb_global_pearson": _pearson_np(pred_rgb[0], target_rgb[0]),
        "rgb_mask_mae": _weighted_mae(rgb_diff, rgb_weights),
        "rgb_mask_rmse": float(np.sqrt(rgb_mask_mse)) if np.isfinite(rgb_mask_mse) else float("nan"),
        "rgb_mask_psnr": _psnr_from_mse(rgb_mask_mse) if np.isfinite(rgb_mask_mse) else float("nan"),
        "rgb_mask_ssim": _ssim_np(rgb_pm, rgb_tm) if rgb_pm is not None and rgb_pm.size > 3 else float("nan"),
        "rgb_mask_pearson": _pearson_np(rgb_pm, rgb_tm) if rgb_pm is not None and rgb_pm.size > 3 else float("nan"),
        "band13_global_mae": float(np.mean(np.abs(band_diff))),
        "band13_global_rmse": float(np.sqrt(band_mse)),
        "band13_global_psnr": _psnr_from_mse(band_mse),
        "band13_global_pearson": _pearson_np(pred_s2[0], target_s2[0]),
        "band13_mask_mae": _weighted_mae(band_diff, band_weights),
        "band13_mask_rmse": float(np.sqrt(band_mask_mse)) if np.isfinite(band_mask_mse) else float("nan"),
        "band13_mask_psnr": _psnr_from_mse(band_mask_mse) if np.isfinite(band_mask_mse) else float("nan"),
        "band13_mask_pearson": _pearson_np(band_pm, band_tm) if band_pm is not None and band_pm.size > 13 else float("nan"),
    }


@torch.no_grad()
def report_evaluation_gan(loader, split_name="test"):
    baseline_eval_model = MultiTemporalResUNet(in_channels=INPUT_CHANNELS, base=CFG.BASE_CHANNELS).to(DEVICE)
    baseline_eval_model.load_state_dict({k: v.to(DEVICE) for k, v in baseline_reference_state.items()}, strict=True)
    baseline_eval_model.eval()
    gan_checkpoint = torch.load(BEST_BLENDED_PATH, map_location=DEVICE)
    model.load_state_dict(state_dict_from_checkpoint(gan_checkpoint), strict=True)
    model.eval()
    per_sample_rows, output_cache = [], {}
    for batch in tqdm(loader, desc=f"GAN report eval {split_name}", leave=False):
        x = batch["x"].to(DEVICE)
        y_np = batch["y"].cpu().numpy()
        t0_t = batch["t0"].to(DEVICE)
        t0_np = batch["t0"].cpu().numpy()
        mask_t = batch["cloud_mask"].to(DEVICE)
        mask_np = batch["cloud_mask"].cpu().numpy()
        sample_ids = batch.get("sample_id", [f"sample_{i}" for i in range(x.shape[0])])
        baseline_raw = baseline_eval_model(x)
        baseline_output = apply_copy_outside_mask(baseline_raw, t0_t, mask_t).cpu().numpy()
        raw_prediction = model(x)
        raw_np = raw_prediction.cpu().numpy()
        gan_blended_output = apply_copy_outside_mask(raw_prediction, t0_t, mask_t)
        blended_np = gan_blended_output.cpu().numpy()
        methods = [
            ("cloudy_t0", t0_np, None),
            ("baseline_output", baseline_output, baseline_eval_checkpoint_path),
            ("gan_raw_prediction", raw_np, BEST_BLENDED_PATH),
            ("gan_blended_output", blended_np, BEST_BLENDED_PATH),
        ]
        for i, sample_id in enumerate(sample_ids):
            output_cache[str(sample_id)] = {
                "t0": t0_np[i],
                "mask": mask_np[i],
                "target": y_np[i],
                "baseline_output": baseline_output[i],
                "gan_raw_prediction": raw_np[i],
                "gan_blended_output": blended_np[i],
            }
            for method, pred_method, ckpt in methods:
                per_sample_rows.append(_per_sample_report_row(method, pred_method[i:i+1], y_np[i:i+1], mask_np[i:i+1], sample_id, split_name, ckpt))
    per_sample_df = pd.DataFrame(per_sample_rows)
    test_ids = per_sample_df.loc[per_sample_df["method"] == "cloudy_t0", "sample_id"].tolist()
    test_hash = _split_hash(test_ids)
    assert test_hash == CFG.EXPECTED_TEST_SAMPLE_SHA256, f"Unexpected test hash: {test_hash}"
    test_hash == CFG.EXPECTED_TEST_SAMPLE_SHA256
    per_sample_df["test_sample_sha256"] = test_hash
    metric_cols = REPORT_METRIC_COLUMNS + ["mask_pixel_count"]
    report_df = per_sample_df.groupby("method", as_index=False)[metric_cols].mean(numeric_only=True)
    report_df.insert(1, "evaluation_split", split_name)
    report_df.insert(2, "split_seed", int(SPLIT_SEED))
    report_df.insert(3, "evaluated_n_samples", len(test_ids))
    report_df.insert(4, "full_test_n_samples", len(test_ids))
    report_df.insert(5, "test_sample_sha256", test_hash)
    report_df.insert(6, "checkpoint", "")
    report_df.insert(7, "mask_threshold", float(EVAL_MASK_THRESHOLD))
    report_df.loc[report_df["method"] == "baseline_output", "checkpoint"] = str(baseline_eval_checkpoint_path)
    report_df.loc[report_df["method"].isin(["gan_raw_prediction", "gan_blended_output"]), "checkpoint"] = str(BEST_BLENDED_PATH)
    id_vars = ["method", "evaluation_split", "split_seed", "evaluated_n_samples", "full_test_n_samples", "test_sample_sha256", "checkpoint", "mask_threshold"]
    long_df = report_df.melt(id_vars=id_vars, value_vars=REPORT_METRIC_COLUMNS, var_name="metric", value_name="value")
    wide_path = CFG.OUT_DIR / "report_metrics_wide.csv"
    long_path = CFG.OUT_DIR / "report_metrics_long.csv"
    per_sample_path = CFG.OUT_DIR / "report_metrics_per_sample.csv"
    json_path = CFG.OUT_DIR / "report_metrics.json"
    ids_path = CFG.OUT_DIR / "final_test_sample_ids_from_evaluation.csv"
    report_df.to_csv(wide_path, index=False)
    long_df.to_csv(long_path, index=False)
    per_sample_df.to_csv(per_sample_path, index=False)
    pd.DataFrame({"sample_id": test_ids}).to_csv(ids_path, index=False)
    row_map = {r["method"]: r for r in report_df.to_dict(orient="records")}
    gan_row = row_map["gan_blended_output"]
    gan_best_checkpoint_is_post_finetune = bool(best_blended_epoch > 0)
    success = {
        "gan_rgb_mask_mae_better_than_user_baseline": bool(gan_row["rgb_mask_mae"] < CFG.BASELINE_RGB_MASK_MAE),
        "gan_band13_mask_mae_better_than_user_baseline": bool(gan_row["band13_mask_mae"] < CFG.BASELINE_BAND13_MASK_MAE),
        "gan_best_checkpoint_is_post_finetune": gan_best_checkpoint_is_post_finetune,
        "best_epoch": int(best_blended_epoch),
        "test_hash_matches_expected": bool(test_hash == CFG.EXPECTED_TEST_SAMPLE_SHA256),
    }
    success["gan_metric_success"] = bool(success["gan_best_checkpoint_is_post_finetune"] and (success["gan_rgb_mask_mae_better_than_user_baseline"] or success["gan_band13_mask_mae_better_than_user_baseline"]))
    success["recommended_method"] = "gan_blended_output" if success["gan_metric_success"] else "baseline_output"
    json_path.write_text(json.dumps(_json_safe({
        "experiment_name": CFG.EXPERIMENT_NAME,
        "split_seed": int(SPLIT_SEED),
        "test_sample_sha256": test_hash,
        "baseline_checkpoint": str(baseline_eval_checkpoint_path),
        "gan_best_blended_checkpoint": str(BEST_BLENDED_PATH),
        "gan_best_raw_checkpoint": str(BEST_RAW_PATH),
        "success_rule": "Model baru hanya boleh diklaim lebih baik jika RGB mask MAE < 0.015089 atau 13-band mask MAE < 0.018551.",
        "success_criteria": success,
        "wide_metrics": report_df.to_dict(orient="records"),
        "files": {"wide": str(wide_path), "long": str(long_path), "per_sample": str(per_sample_path), "sample_ids": str(ids_path)},
    }), indent=2), encoding="utf-8")
    display(report_df[id_vars + REPORT_METRIC_COLUMNS])
    return report_df, long_df, per_sample_df, output_cache, success


report_metrics_df, report_metrics_long_df, report_metrics_per_sample_df, visual_output_cache, gan_success_criteria = report_evaluation_gan(test_loader, split_name="test")


## 10. Visual Random/Heavy/Best/Worst dan Audit Report

In [ ]:
# Visual random/heavy/best/worst dan laporan GAN.

def rgb_display(arr):
    return np.clip(np.transpose(np.asarray(arr)[list(RGB_BAND_INDICES)], (1, 2, 0)), 0.0, 1.0)


def mask_display(mask):
    mask = np.asarray(mask)
    return np.clip(mask[0] if mask.ndim == 3 else mask, 0.0, 1.0)


def error_map_mask_only(pred, target, mask):
    err = np.mean(np.abs(np.asarray(pred)[list(RGB_BAND_INDICES)] - np.asarray(target)[list(RGB_BAND_INDICES)]), axis=0)
    return err * (mask_display(mask) >= EVAL_MASK_THRESHOLD)


def select_visual_sample_ids(per_sample_df):
    final_df = per_sample_df[per_sample_df["method"] == "gan_blended_output"].copy().sort_values("sample_id")
    sample_ids = final_df["sample_id"].tolist()
    rng = random.Random(SEED)
    return {
        "random_samples": rng.sample(sample_ids, min(4, len(sample_ids))),
        "heavy_cloud_samples": final_df.sort_values("mask_pixel_count", ascending=False)["sample_id"].head(4).tolist(),
        "best_mae_samples": final_df.sort_values("rgb_mask_mae", ascending=True)["sample_id"].head(4).tolist(),
        "worst_mae_samples": final_df.sort_values("rgb_mask_mae", ascending=False)["sample_id"].head(4).tolist(),
    }


def save_gan_visual_grid(output_cache, sample_ids, out_path, title_prefix):
    columns = ["Input cloudy RGB", "Soft cloud mask", "Pseudo ground truth RGB", "Baseline output", "GAN raw prediction", "GAN blended output", "Error map mask-only"]
    fig, axes = plt.subplots(len(sample_ids), len(columns), figsize=(19, max(3, 2.8 * len(sample_ids))))
    if len(sample_ids) == 1:
        axes = axes[None, :]
    for row_i, sample_id in enumerate(sample_ids):
        item = output_cache[sample_id]
        panels = [
            (rgb_display(item["t0"]), None),
            (mask_display(item["mask"]), "gray"),
            (rgb_display(item["target"]), None),
            (rgb_display(item["baseline_output"]), None),
            (rgb_display(item["gan_raw_prediction"]), None),
            (rgb_display(item["gan_blended_output"]), None),
            (error_map_mask_only(item["gan_blended_output"], item["target"], item["mask"]), "magma"),
        ]
        for col_i, (image, cmap) in enumerate(panels):
            ax = axes[row_i, col_i]
            ax.imshow(image, cmap=cmap, vmin=0, vmax=1) if cmap else ax.imshow(image)
            if row_i == 0:
                ax.set_title(columns[col_i], fontsize=9)
            if col_i == 0:
                ax.set_ylabel(sample_id, fontsize=8)
            ax.set_xticks([])
            ax.set_yticks([])
    fig.suptitle(title_prefix, fontsize=12)
    plt.tight_layout()
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=160, bbox_inches="tight")
    plt.close(fig)
    return out_path


visual_categories = select_visual_sample_ids(report_metrics_per_sample_df)
visual_categories_path = CFG.OUT_DIR / "visual_selection_categories.json"
visual_categories_path.write_text(json.dumps(visual_categories, indent=2), encoding="utf-8")
visual_random_test = save_gan_visual_grid(visual_output_cache, visual_categories["random_samples"], CFG.OUT_DIR / "visual_random_test.png", "GAN random test samples")
visual_heavy_cloud_test = save_gan_visual_grid(visual_output_cache, visual_categories["heavy_cloud_samples"], CFG.OUT_DIR / "visual_heavy_cloud_test.png", "GAN heavy cloud test samples")
visual_best_mae = save_gan_visual_grid(visual_output_cache, visual_categories["best_mae_samples"], CFG.OUT_DIR / "visual_best_mae.png", "GAN best RGB mask MAE samples")
visual_worst_mae = save_gan_visual_grid(visual_output_cache, visual_categories["worst_mae_samples"], CFG.OUT_DIR / "visual_worst_mae.png", "GAN worst RGB mask MAE samples")


def _row_for(method):
    return report_metrics_df[report_metrics_df["method"] == method].iloc[0].to_dict()


def _fmt(value):
    return "nan" if value is None or not np.isfinite(float(value)) else f"{float(value):.6f}"


cloudy_row = _row_for("cloudy_t0")
baseline_row = _row_for("baseline_output")
raw_row = _row_for("gan_raw_prediction")
gan_row = _row_for("gan_blended_output")
if gan_success_criteria["gan_metric_success"]:
    success_text = "GAN memenuhi ambang klaim lebih baik dengan checkpoint hasil fine-tune."
else:
    success_text = "Baseline tetap model utama jika GAN raw/blended belum melewati ambang RGB mask MAE < 0.015089 atau 13-band mask MAE < 0.018551."

metrics_table = [
    "| Method | RGB global MAE | RGB mask MAE | 13-band global MAE | 13-band mask MAE | RGB global PSNR | 13-band global PSNR |",
    "|---|---:|---:|---:|---:|---:|---:|",
]
for method, row in [
    ("cloudy_t0", cloudy_row),
    ("baseline_output", baseline_row),
    ("gan_raw_prediction", raw_row),
    ("gan_blended_output", gan_row),
]:
    metrics_table.append(f"| {method} | {_fmt(row['rgb_global_mae'])} | {_fmt(row['rgb_mask_mae'])} | {_fmt(row['band13_global_mae'])} | {_fmt(row['band13_mask_mae'])} | {_fmt(row['rgb_global_psnr'])} | {_fmt(row['band13_global_psnr'])} |")

report_lines = [
    "# GAN_TRAINING_REPORT",
    "",
    f"Tanggal run: {datetime.now().isoformat(timespec='seconds')}",
    "",
    "## Scope GAN",
    "",
    "- Discriminator GAN bersifat conditional dan menilai raw_prediction, bukan blended output.",
    "- Generator MultiTemporalResUNet diinisialisasi dari checkpoint Baseline `best_multitemporal_resunet_hardmask.pth`, bukan random initialization.",
    "- Discriminator conditional menerima concat(cloudy_s2_t0 13, mask_t0 1, SAR_t0 VV/VH 2, image 13) = 29 channel dengan Spectral Normalization.",
    f"- Best checkpoint dipilih berdasarkan `val_model_cloud_mae`.",
    f"- Best blended epoch: `{gan_success_criteria['best_epoch']}`.",
    f"- Test sample SHA256: `{report_metrics_df['test_sample_sha256'].iloc[0]}`",
    "",
    "## Hyperparameter",
    "",
    f"- Epoch fine-tune: {CFG.GAN_EPOCHS}",
    f"- LR generator: {CFG.G_LR}",
    f"- LR discriminator: {CFG.D_LR}",
    f"- lambda_adv: {CFG.LAMBDA_ADV}",
    f"- lambda_fm: {CFG.LAMBDA_FEATURE_MATCHING}",
    f"- lambda_ssim: {CFG.LAMBDA_SSIM}",
    "",
    "## Metrik Evaluasi",
    "",
    *metrics_table,
    "",
    "## Aturan Klaim",
    "",
    "- RGB mask MAE < 0.015089, atau",
    "- 13-band mask MAE < 0.018551.",
    "",
    "## Audit Wajib",
    "",
    "- Penamaan model dan output menggunakan istilah umum GAN.",
    "- Discriminator sudah conditional 29 channel.",
    "- Discriminator menilai `raw_prediction`, bukan `gan_blended_output`.",
    "- Test sample SHA256 sama dengan Baseline.",
    "- Jika checkpoint terbaik masih epoch 0, laporan menyatakan training adversarial belum menghasilkan checkpoint setelah warm-start.",
    "- Raw prediction dan blended output dievaluasi terpisah.",
    f"- Apakah checkpoint terbaik masih epoch 0 atau berubah: best blended epoch = `{gan_success_criteria['best_epoch']}`.",
    f"- Perbandingan raw_prediction dengan Baseline: raw RGB mask MAE = `{_fmt(raw_row['rgb_mask_mae'])}`, Baseline RGB mask MAE = `{_fmt(baseline_row['rgb_mask_mae'])}`.",
    f"- Perbandingan gan_blended_output dengan Baseline: blended RGB mask MAE = `{_fmt(gan_row['rgb_mask_mae'])}`, Baseline RGB mask MAE = `{_fmt(baseline_row['rgb_mask_mae'])}`.",
    f"- Status GAN terhadap kriteria Baseline: `{'ya' if gan_success_criteria['gan_metric_success'] else 'tidak'}`.",
    f"- Rekomendasi laporan TA: `{'gan_blended_output dapat dipertimbangkan sebagai output utama, dengan catatan raw prediction tetap dilaporkan terpisah' if gan_success_criteria['gan_metric_success'] else 'Baseline ResUNet tetap model utama; GAN dilaporkan sebagai eksperimen adversarial'}`.",
    "",
    "## Kesimpulan",
    "",
    f"- {success_text}",
    f"- Rekomendasi metrik: `{gan_success_criteria['recommended_method']}`",
    "",
    "## Artefak",
    "",
    f"- report_metrics.json: `{CFG.OUT_DIR / 'report_metrics.json'}`",
    f"- report_metrics_wide.csv: `{CFG.OUT_DIR / 'report_metrics_wide.csv'}`",
    f"- report_metrics_long.csv: `{CFG.OUT_DIR / 'report_metrics_long.csv'}`",
    f"- report_metrics_per_sample.csv: `{CFG.OUT_DIR / 'report_metrics_per_sample.csv'}`",
    f"- training_history.csv: `{TRAINING_HISTORY_PATH}`",
    f"- epoch0_baseline_init.pth: `{EPOCH0_PATH}`",
    f"- best_val_blended_cloud_mae.pth: `{BEST_BLENDED_PATH}`",
    f"- best_val_raw_cloud_mae.pth: `{BEST_RAW_PATH}`",
    f"- last_gan.pth: `{GAN_LAST_PATH}`",
    f"- visual_random_test.png: `{visual_random_test}`",
    f"- visual_heavy_cloud_test.png: `{visual_heavy_cloud_test}`",
    f"- visual_best_mae.png: `{visual_best_mae}`",
    f"- visual_worst_mae.png: `{visual_worst_mae}`",
]
report_path = CFG.OUT_DIR / "GAN_TRAINING_REPORT.md"
report_path.write_text("\n".join(report_lines) + "\n", encoding="utf-8")
print("Saved GAN report:", report_path)
print("\n".join(report_lines[:55]))
